In [3]:
!pip install torch transformers datasets accelerate bitsandbytes peft unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 5.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 

In [17]:
%%writefile main.py
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, TrainerCallback
from peft import LoraConfig
from transformers import DataCollatorForLanguageModeling
import os
import pandas as pd


os.environ["WANDB_DISABLED"] = "true"

# 1. Load LLaMA-2 7B in 4-bit mode
model_name = "meta-llama/Llama-2-7b-hf"
max_seq_length = 512  # Maximum sequence length

# Load model in 4-bit quantized mode
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    load_in_4bit=True,
    max_seq_length=max_seq_length
)

# 2. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # LoRA rank (controls trainable parameter count)
    lora_alpha=32,  # Scaling factor for LoRA
    lora_dropout=0.05,  # Dropout rate to prevent overfitting
    bias="none",  # No bias training
    use_rslora=False,  # Keep standard LoRA
)

# Print trainable parameters
model.print_trainable_parameters()

# 3. Load and Tokenize Dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_seq_length)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Use subset of the dataset for quicker training
small_train_dataset = tokenized_datasets["train"].select(range(100))  # Use first 100 examples
small_eval_dataset = tokenized_datasets["validation"].select(range(10))  # Use first 10 examples

# 4. Training Setup
training_args = TrainingArguments(
    output_dir="./llama-unsloth",
    per_device_train_batch_size=20,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    num_train_epochs=1,  # Train for just 1 epoch
    save_strategy="epoch",
    optim="adamw_torch",
    fp16=True,  # Enables mixed precision training
    logging_steps=10,  # Log every 10 steps
)

# data collator for Causal LM training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Set to False for autoregressive models like LLaMA
)


# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    data_collator=data_collator,  # Add this to correctly handle labels
)

# 6. Train the Model
trainer.train()

# 7. Return the trainer object for testing
def get_trainer():
    return trainer

Overwriting main.py


In [2]:
!python main.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2025-02-22 03:13:04.289851: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740193984.364970   10133 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740193984.396996   10133 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-22 03:13:04.517302: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
🦥 Unsloth Zoo will now patch everything to m